# v6 overlap40 · SwinV2-Base · seed1337
全量微调、Full五通道、physical batch=4。仅使用Train/Val选模，**不读取Test**。

In [ ]:
from pathlib import Path
import hashlib, importlib, importlib.metadata, importlib.util, json, os, subprocess, sys

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
CONFIG_FILE = 'v6_overlap40_stdl_swinv2_base_full_seed1337.json'

required = [('rasterio', 'rasterio'), ('timm', 'timm'), ('segmentation_models_pytorch', 'segmentation-models-pytorch==0.5.0')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0' and 'segmentation-models-pytorch==0.5.0' not in missing:
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
CONFIG_PATH = PROJECT_DIR / 'configs' / CONFIG_FILE
assert CONFIG_PATH.is_file(), f'配置不存在，请先更新仓库: {CONFIG_PATH}'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['module'] == 'stdl_swinv2_base' and config['seed'] == 1337
assert config['automatic_test_evaluation'] is False
print('Commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

data_paths = [p for p in Path('/kaggle/input').rglob('dataset_v6_random811_overlap40') if p.is_dir() and (p / 'dataset_protocol.json').is_file()]
assert data_paths, '没有找到dataset_v6_random811_overlap40，请先Add Input'
DATA_ROOT = data_paths[0]
for filename, expected in config['expected_metadata_sha256'].items():
    assert sha256(DATA_ROOT / filename) == expected, f'数据元数据不匹配: {filename}'
weight_paths = [p for p in Path('/kaggle/input').rglob(config['pretrain_filename']) if p.is_file()]
matching_weights = [p for p in weight_paths if sha256(p) == config['expected_pretrain_sha256']]
assert matching_weights, f"没有找到正确的预训练权重: {config['pretrain_filename']}"
PRETRAIN_DIR = matching_weights[0].parent
print('数据:', DATA_ROOT)
print('预训练权重:', matching_weights[0])
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")

In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'run_autodl_stdl_swin.py'),
    '--project-dir', str(PROJECT_DIR), '--config', str(CONFIG_PATH),
    '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
    '--pretrain-dir', str(PRETRAIN_DIR),
]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert metrics['seed'] == 1337 and metrics['selection_metric'] == 'val_mIoU_fg'
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载压缩包:', Path(str(result_dir) + '.zip'))